# 🤝 Ensemble Yöntemleri (Stacking ve Voting)

Birden fazla modelin gücünü birleştirerek (Ensemble) daha güçlü ve dayanıklı tahminler üretme şablonları. Özellikle Kaggle'da sıralama yükseltmek için kritik öneme sahiptir.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.ensemble import VotingRegressor, VotingClassifier
from sklearn.ensemble import StackingRegressor, StackingClassifier
from sklearn.linear_model import Ridge, LogisticRegression
import xgboost as xgb
import lightgbm as lgb

## 1. Voting Ensemble (Basit Oylama / Ortalama)
Farklı modellerin (örn: XGBoost ve LightGBM) tahminlerini eşit ağırlıkta toplar veya ortalamasını alır. Basittir ama hataları dengeler.

In [ ]:
def train_voting_ensemble(X_train, y_train, X_test, task='regression'):
    """
    İki farklı güçlü modeli (LightGBM ve XGBoost) birleştirir.
    """
    if task == 'regression':
        model1 = lgb.LGBMRegressor(random_state=42)
        model2 = xgb.XGBRegressor(random_state=42)
        
        ensemble = VotingRegressor(estimators=[
            ('lgb', model1),
            ('xgb', model2)
        ])
    else:
        # Sınıflandırma için 'soft' voting (olasılıkların ortalaması) genellikle daha iyidir
        model1 = lgb.LGBMClassifier(random_state=42)
        model2 = xgb.XGBClassifier(random_state=42)
        
        ensemble = VotingClassifier(estimators=[
            ('lgb', model1),
            ('xgb', model2)
        ], voting='soft')
        
    ensemble.fit(X_train, y_train)
    preds = ensemble.predict(X_test)
    
    return ensemble, preds

## 2. Stacking (İleri Seviye Model Birleştirme)
Birden fazla 'Temel Model' (Base Model) tahmin yapar. Bu tahminler yeni bir özellik (feature) gibi kullanılarak bir 'Meta Model' (Final Model) eğitilir. 
Genellikle Meta Model olarak basit lineer modeller (Ridge Regresyon veya Lojistik Regresyon) seçilir.

In [ ]:
def train_stacking_ensemble(X_train, y_train, X_test, task='regression'):
    """
    Ağaç tabanlı güçlü modellerin tahminlerini alıp, basit bir Meta Model (Ridge/Logistic) ile harmanlar.
    """
    if task == 'regression':
        base_models = [
            ('lgb', lgb.LGBMRegressor(random_state=42)),
            ('xgb', xgb.XGBRegressor(random_state=42))
        ]
        meta_model = Ridge(alpha=10.0)
        
        stacking = StackingRegressor(estimators=base_models, final_estimator=meta_model, cv=5)
        
    else:
        base_models = [
            ('lgb', lgb.LGBMClassifier(random_state=42)),
            ('xgb', xgb.XGBClassifier(random_state=42))
        ]
        meta_model = LogisticRegression()
        
        stacking = StackingClassifier(estimators=base_models, final_estimator=meta_model, cv=5)
        
    # Stacking, cross-validation mantığıyla çalışır (fit sırasında biraz uzun sürebilir)
    stacking.fit(X_train, y_train)
    preds = stacking.predict(X_test)
    
    return stacking, preds